# Kannada-MNIST

## Experiment and submission report

Classify Kannada digits from 28 × 28 grayscale images. Compare a TRAIN-only CNN with a CNN that also learns from Dig-MNIST, then refit the model on all labeled data for final submission.

- [Competition](https://www.kaggle.com/competitions/Kannada-MNIST)
- [Data](https://www.kaggle.com/competitions/Kannada-MNIST/data)
- [Baseline experiment](experiment-baseline.ipynb)
- [Mixed experiment](experiment-mixed.ipynb)
- [Full-data submission model](submission.ipynb)

This report contains one completed local run per experiment and three confirmed competition submissions. Queued, canceled, and otherwise unfinished Kaggle runs are excluded from result tables and plots. The 2021 competition entry is outside this experiment series and is also excluded.

Experimental validation tables and plots use **local runs only**, with one selected result per method. Kaggle notebook-run validation metrics are excluded. The separate competition-score section contains actual scored submissions; it does not add another validation run for an experiment.

## Methods

All three methods use two convolution/ReLU/max-pooling blocks (16 and 32 channels), a 128-unit dense layer, and ten output logits. Pixels are divided by 255. Adam uses learning rate 0.001, batch size 256, model/DataLoader seed 42, and ten fixed epochs. No augmentation, early stopping, or best-checkpoint selection is used.

| Notebook | TRAIN fitting rows | Dig-MNIST fitting rows | TRAIN holdout | Dig-MNIST holdout |
| --- | ---: | ---: | ---: | ---: |
| `experiment-baseline.ipynb` | 48,000 | 0 | 12,000 | 2,048 |
| `experiment-mixed.ipynb` | 48,000 | 8,192 | 12,000 | 2,048 |
| `submission.ipynb` | 60,000 | 10,240 | None | None |

TRAIN and Dig-MNIST are independently split 80/20 with scikit-learn stratification by label and split seed 42. Both experiments use identical row positions from each original CSV. Baseline leaves the 8,192 Dig learning rows unused; mixed concatenates them with TRAIN learning rows. No competition test image is used for fitting.

Mixed has more optimizer updates per epoch because it contains more images. This is a comparison at equal epochs, not equal compute. Dig-MNIST represents a handwriting/domain shift; report its validation performance separately from TRAIN validation.

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.io as pio
from IPython.display import display

pio.renderers.default = "notebook"
pio.templates.default = "plotly_white"


## Shared CNN architecture

Baseline, Mixed, and the full-data submission use the same CNN; their training data differs. Define the actual PyTorch model below and let `torchinfo` compute each layer's input/output shape and parameter count from a sample input of shape **(batch=1, channels=1, height=28, width=28)**.

The summary runs on CPU with newly initialized weights. It does not train the model or require a saved checkpoint, and weight values do not affect the architecture summary. The network has **206,922 trainable parameters**.

Each 3 × 3 convolution uses stride 1 and padding 1. Each 2 × 2 max-pool halves height and width. The final linear layer produces ten logits for `CrossEntropyLoss`; prediction code applies softmax afterward. No dropout or batch normalization is used.


In [ ]:
from importlib.metadata import version

from torch import nn
from torchinfo import summary

model = nn.Sequential(
    nn.Conv2d(1, 16, 3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Conv2d(16, 32, 3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Flatten(),
    nn.Linear(32 * 7 * 7, 128),
    nn.ReLU(),
    nn.Linear(128, 10),
)

print("torchinfo:", version("torchinfo"))
model_summary = summary(
    model,
    input_size=(1, 1, 28, 28),
    col_names=("input_size", "output_size", "num_params"),
    device="cpu",
    mode="eval",
    verbose=1,
)
assert model_summary.total_params == 206922


## Completed local experiments

The embedded snapshot below comes from each completed ten-epoch run's `output/<experiment>/evaluation_summary.csv`. It is stored in notebook source so the report does not depend on ignored local artifacts, credentials, or a Kaggle connection. Precision, recall, and F1 are macro averages across digits 0–9; accuracy is sample accuracy. Both experiments evaluate the final epoch.

Baseline and Mixed each contribute exactly one completed local ten-epoch run. Re-running the same method on Kaggle does not create a second entry in this comparison.

In [ ]:
results = pd.DataFrame(
    [
        {
            "experiment": "Baseline",
            "dataset": "Validation",
            "samples": 12000,
            "accuracy": 0.9926666666666668,
            "precision": 0.9927401630155864,
            "recall": 0.9926666666666668,
            "f1": 0.9926711226188416,
        },
        {
            "experiment": "Baseline",
            "dataset": "Dig-MNIST validation",
            "samples": 2048,
            "accuracy": 0.724609375,
            "precision": 0.7644917960041635,
            "recall": 0.7246293639406982,
            "f1": 0.723925958844546,
        },
        {
            "experiment": "Mixed",
            "dataset": "Validation",
            "samples": 12000,
            "accuracy": 0.9935833333333334,
            "precision": 0.993597359617229,
            "recall": 0.9935833333333332,
            "f1": 0.9935838715090616,
        },
        {
            "experiment": "Mixed",
            "dataset": "Dig-MNIST validation",
            "samples": 2048,
            "accuracy": 0.9482421875,
            "precision": 0.9490404685160816,
            "recall": 0.9482568149210904,
            "f1": 0.948462985198388,
        },
    ]
)

split_provenance = {
    "seed": 42,
    "validation_fraction": 0.2,
    "stratify_column": "label",
    "row_order": "original CSV order",
    "train_csv_sha256": "4425af3e09e2f074f5f3c80e7ed8f8e86214d148f14c979103c2143180fc1f95",
    "dig_csv_sha256": "a9b3cac459416289a1d1e94f6536988a11e92abcf8c2ddf360bb8f8bdf697f54",
    "scikit_learn_version": "1.6.1",
}

metric_columns = ["accuracy", "precision", "recall", "f1"]
assert results[metric_columns].ge(0).all().all()
assert results[metric_columns].le(1).all().all()
display(results.style.format({metric: "{:.4%}" for metric in metric_columns}))
print("Shared input and split provenance:")
display(pd.Series(split_provenance, name="value").to_frame())


In [ ]:
chart_data = results.melt(
    id_vars=["experiment", "dataset", "samples"],
    value_vars=metric_columns,
    var_name="metric",
    value_name="score",
)
fig = px.bar(
    chart_data,
    x="metric",
    y="score",
    color="experiment",
    facet_col="dataset",
    barmode="group",
    hover_data=["samples"],
    text_auto=".2%",
    title="Completed local experiments on identical held-out rows",
    labels={"metric": "Metric", "score": "Score", "experiment": "Experiment"},
)
fig.update_yaxes(range=[0, 1.08], tickformat=".0%")
fig.update_layout(height=460, legend_title_text="Experiment")
fig.show()


## Interpretation

Mixed raises Dig holdout accuracy from **72.4609% to 94.8242%**, a gain of **22.3633 percentage points**. TRAIN holdout accuracy changes from **99.2667% to 99.3583%**, a gain of **0.0917 percentage points**. The large Dig improvement supports learning from the second handwriting source, while the much smaller TRAIN difference should not be treated as a reliable gain from a single run.

These are validation measurements, not estimates guaranteed to match the hidden competition distribution. Repeated experiment selection can overfit the holdouts. The final full-data model uses these former holdouts for fitting and therefore cannot reuse their scores as unbiased evaluation of that model.

## Confirmed Kaggle competition scores

All three entries below completed scoring and were verified through the Kaggle submissions API on 2026-11-17 UTC. The table records the exact submission IDs and timestamps; the links identify the submitted notebook versions. Public and private scores are accuracy on separate competition partitions, not local validation scores.

**Baseline v3 predates the shared Dig holdout experiment versions.** It trained on the 80% TRAIN learning subset without Dig fitting. The canceled baseline v4 has no completed score and is excluded. Mixed v1 is the newly recreated notebook using both 80% learning subsets. Full-data v1 uses all labeled rows.

- [Baseline v3 (TRAIN-only)](https://www.kaggle.com/code/pocariblitz/kannada-mnist-baseline?scriptVersionId=370668060) — submission `58469292`
- [Mixed v1 (80% TRAIN + 80% Dig)](https://www.kaggle.com/code/pocariblitz/kannada-mnist-mixed?scriptVersionId=370695751) — submission `58472256`
- [Full-data v1 (100% TRAIN + 100% Dig)](https://www.kaggle.com/code/pocariblitz/kannada-mnist-submission?scriptVersionId=370693111) — submission `58471749`

In [ ]:
competition_scores = pd.DataFrame(
    [
        {
            "method": "Baseline v3 (TRAIN-only)",
            "submission_id": 58469292,
            "submitted_utc": "2026-11-17T12:57:41.037Z",
            "public_accuracy": 0.9662,
            "private_accuracy": 0.9646,
        },
        {
            "method": "Mixed v1 (80% TRAIN + 80% Dig)",
            "submission_id": 58472256,
            "submitted_utc": "2026-11-17T14:45:12.300Z",
            "public_accuracy": 0.968,
            "private_accuracy": 0.9678,
        },
        {
            "method": "Full-data v1 (100% TRAIN + 100% Dig)",
            "submission_id": 58471749,
            "submitted_utc": "2026-11-17T14:25:51.487Z",
            "public_accuracy": 0.9612,
            "private_accuracy": 0.9642,
        },
    ]
)
display(
    competition_scores.style.format(
        {"public_accuracy": "{:.5f}", "private_accuracy": "{:.5f}"}
    )
)
competition_chart = competition_scores.melt(
    id_vars="method",
    value_vars=["public_accuracy", "private_accuracy"],
    var_name="partition",
    value_name="accuracy",
)
fig = px.bar(
    competition_chart,
    x="method",
    y="accuracy",
    color="partition",
    barmode="group",
    text_auto=".5f",
    title="Completed competition submissions",
    labels={
        "method": "Submitted model",
        "accuracy": "Accuracy",
        "partition": "Competition partition",
    },
)
fig.update_yaxes(range=[0, 1.08], tickformat=".0%")
fig.show()


## Final submission method and outcome

`submission.ipynb` initializes a fresh CNN and trains for ten epochs on all 70,240 labeled images. It saves weights, input hashes, software versions, training history, and predictions for `test.csv`. Original test IDs and the `id,label` schema are preserved. No unlabeled test image is used for training.

The Kaggle full-data v1 execution and its competition submission both completed. Its public accuracy is **0.96120** and private accuracy is **0.96420**. No held-out accuracy or F1 is available for this fit, because every labeled image is used for training. Training accuracy describes optimization rather than generalization and is not included in the validation comparison.

**The mixed experiment currently has the highest confirmed competition score in this experiment series:** public **0.96800**, private **0.96780**. Full-data fitting is lower by **0.00680** public and **0.00360** private (0.68 and 0.36 percentage points). Full-data fitting also falls slightly below baseline v3 on both partitions. More labeled training data did not improve the measured score in this single run.

These results do not isolate the reason for the difference. Changing the training set also changes the number and order of optimizer updates; a fixed seed does not make different datasets follow the same training trajectory. Repeated seeds or additional model selection experiments would be needed to investigate. Do not present training accuracy as evidence that the full-data model generalizes better.

Kaggle notebook execution and competition submission are separate steps. No new run, upload, or competition submission is triggered by this report.
